<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/IndexTTS_2_5_Colab_L4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndexTTS 2.5 — Google Colab Pro（NVIDIA L4）

这份 Notebook 会在 Colab 的 `/content` 临时磁盘中安装并运行官方 **IndexTTS 2.5**：

- 检查运行时是否为 **NVIDIA L4**
- 克隆官方仓库并用 `uv` 安装 WebUI 依赖
- 从 Hugging Face 下载 `IndexTeam/IndexTTS-2.5` 权重
- 使用 **BF16** 在 L4 上执行一次中文语音合成测试
- 启动带临时公网链接的 Gradio WebUI

> **开始前：** 在 Colab 菜单选择 **代码执行程序 → 更改运行时类型 → L4 GPU**。然后按顺序运行全部单元格。首次安装和下载模型通常需要较长时间。

> **合规提示：** 仅克隆你本人或已获明确授权的声音；不要用于冒充、欺诈或绕过身份验证。模型与代码受其各自许可证约束。


In [ ]:
# 1. 检查 Colab GPU；不是 L4 时立即停止，避免装错运行时
import os, subprocess

assert os.path.exists('/content'), '请在 Google Colab 中运行此 Notebook。'
try:
    gpu_name = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        text=True
    ).strip()
except Exception as exc:
    raise RuntimeError('未检测到 NVIDIA GPU。请在“更改运行时类型”中选择 L4 GPU。') from exc

print('检测到 GPU：', gpu_name)
assert 'L4' in gpu_name, f'当前不是 L4：{gpu_name}。请切换为 L4 GPU 后重新运行。'
print('✅ NVIDIA L4 检查通过')
!nvidia-smi


In [ ]:
# 2. 安装基础工具
%cd /content
!python -m pip install -q -U uv huggingface_hub hf_xet
!uv --version


In [ ]:
# 3. 获取官方 IndexTTS 2.5 源码
# 重新运行本格会更新已有仓库；不会删除已下载的 checkpoints。
from pathlib import Path
import subprocess

repo = Path('/content/index-tts')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/index-tts/index-tts.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)

commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('官方仓库 commit:', commit)
%cd /content/index-tts


In [ ]:
# 4. 安装项目依赖（WebUI；不安装非必要的 DeepSpeed）
# uv 会按官方锁文件创建独立 .venv。首次执行可能需要数分钟。
%cd /content/index-tts
!uv sync --extra webui


In [ ]:
# 5. 下载 IndexTTS 2.5 模型权重到 checkpoints/
# 若 Hugging Face 要求授权，可在下一格使用 Token；公开模型通常无需 Token。
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path('/content/index-tts/checkpoints')
model_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id='IndexTeam/IndexTTS-2.5',
    local_dir=str(model_dir),
    resume_download=True,
)
assert (model_dir / 'config.yaml').exists(), '模型下载不完整：缺少 checkpoints/config.yaml'
print('✅ 模型已下载：', model_dir)


### 如果上格出现 Hugging Face 401/403

仅在模型页面要求登录或接受许可证时运行下一格。Token 输入不会显示；不要把 Token 写死在 Notebook 中。先在 Hugging Face 创建一个只读 Token。


In [ ]:
# 可选：使用 Hugging Face 只读 Token 重试下载
# 不需要时跳过本格。
from getpass import getpass
from huggingface_hub import login, snapshot_download

hf_token = getpass('Hugging Face 只读 Token：')
login(token=hf_token, add_to_git_credential=False)
snapshot_download(
    repo_id='IndexTeam/IndexTTS-2.5',
    local_dir='/content/index-tts/checkpoints',
    token=hf_token,
)

print('✅ 授权下载完成')


In [ ]:
# 6. 检查项目环境和 GPU 加速
%cd /content/index-tts
!uv run tools/gpu_check.py
!uv run python -c "import torch; print('torch=', torch.__version__); print('cuda=', torch.cuda.is_available()); print('gpu=', torch.cuda.get_device_name(0)); print('bf16=', torch.cuda.is_bf16_supported())"


In [ ]:
# 7. 下载官方示例音频，并用 BF16 做一次端到端测试
# 首次加载模型会占用一些时间。
%cd /content/index-tts
!uv run python -c "from indextts.utils.examples_downloader import ensure_examples_available; ensure_examples_available()"

test_script = r'''
from indextts.infer_v2_5 import IndexTTS2

tts = IndexTTS2(
    cfg_path='checkpoints/config.yaml',
    model_dir='checkpoints',
    use_bf16=True,
)
tts.infer(
    spk_audio_prompt='examples/voice_01.wav',
    text='大家好，这是 IndexTTS 二点五在 Google Colab L4 显卡上的测试。',
    lang='ZH',
    output_path='output_demo.wav',
    duration_factor=1.0,
    verbose=True,
)
print('生成完成：/content/index-tts/output_demo.wav')
'''
open('/content/index-tts/colab_test.py', 'w', encoding='utf-8').write(test_script)
!PYTHONPATH="$PYTHONPATH:." uv run colab_test.py

from IPython.display import Audio, display
display(Audio('/content/index-tts/output_demo.wav'))


## 启动 WebUI

下面会复制一份 `webui.py` 并只修改 Gradio 启动参数，使其生成临时公网链接。运行后：

1. 等待输出 `Running on public URL: https://...gradio.live`；
2. 点击该链接；
3. 在 WebUI 中上传已获授权的参考音频并输入文本；
4. **保持最后一个单元格运行**，链接才会持续有效。

L4 推荐使用 **BF16**。如果界面提供 BF16 选项，请保持开启；DeepSpeed 和编译 CUDA Kernel 先关闭，稳定运行后再测试。Gradio 临时链接不是永久部署，Colab 会话结束后即失效。


In [ ]:
# 8. 为 Colab 创建 WebUI 启动副本（不改官方原文件）
from pathlib import Path

src = Path('/content/index-tts/webui.py')
dst = Path('/content/index-tts/webui_colab.py')
text = src.read_text(encoding='utf-8')
old = 'demo.launch(server_name=cmd_args.host, server_port=cmd_args.port)'
new = 'demo.launch(server_name=cmd_args.host, server_port=cmd_args.port, share=True, show_error=True)'
if old not in text:
    raise RuntimeError('官方 webui.py 的启动代码已变化。请查看上一格记录的 commit，并按最新参数调整。')
dst.write_text(text.replace(old, new, 1), encoding='utf-8')
print('✅ 已创建：', dst)


In [ ]:
# 9. 启动 IndexTTS 2.5 WebUI
# 本格会持续运行；停止本格或断开 Colab 后，公网链接会失效。
%cd /content/index-tts
!PYTHONPATH="$PYTHONPATH:." uv run webui_colab.py --host 0.0.0.0 --port 7860


## 常见问题

- **不是 L4：** 重新选择“代码执行程序 → 更改运行时类型 → L4 GPU”，再从第一格开始。
- **CUDA Out of Memory：** 停止 WebUI，选择“断开并删除运行时”，重新连接 L4；避免同时运行测试模型和 WebUI。
- **Gradio 链接失效：** 重新运行最后一格，获取新的 `gradio.live` 地址。
- **模型下载中断：** 重新运行模型下载格；下载器会复用已完成文件。
- **Colab 重启后文件消失：** `/content` 是临时磁盘，需要重新安装和下载。可将成品音频及时下载到本地或复制到 Google Drive。
- **首次运行较慢：** 依赖安装、权重下载和首次模型加载都需要时间；后续同一会话内会明显更快。
